# Actividad 1 · Correlación y regresión lineal simple

**Recorrido elegido:** SQL + Python  
**Tiempo estimado:** 1 pomodoro (20 minutos)  
**Carácter:** obligatoria.

> Realizá solamente uno de los dos recorridos equivalentes: **SQL + Python** o **SQL + R**.

En la Unidad N° 5 viste que los problemas supervisados pueden ser de **clasificación** o de **regresión**. Hasta ahora construiste un modelo que predice una categoría. En esta actividad vas a trabajar con la otra lógica: **estimar un valor numérico**.

El caso será deliberadamente sencillo: analizaremos la relación entre la **superficie de una vivienda** y su **precio**, para comprender cómo se conectan la correlación de Pearson, la regresión lineal simple, las predicciones, los residuos y el coeficiente de determinación \(R^2\).

El notebook es **autocontenido**: genera internamente los datos y no requiere archivos auxiliares.

## 1. Preparar el entorno y los datos

Ejecutá la siguiente celda. Se generará un pequeño conjunto de viviendas y se cargará en una base SQLite en memoria para poder consultarlo con SQL.

In [ ]:
#@title Preparación automática del entorno y de los datos (ejecutar sin modificar)
import importlib.util
import subprocess
import sys
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if importlib.util.find_spec("sklearn") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

i = np.arange(1, 61)
superficie = np.round(42 + 2.05 * i + 4 * np.sin(i * 0.65), 1)
ruido = 13 * np.sin(i * 1.37) + 7 * np.cos(i * 0.48)
precio = np.round(38 + 1.82 * superficie + ruido, 1)

viviendas = pd.DataFrame({
    "id": i,
    "superficie_m2": superficie,
    "precio_miles_usd": precio
})

con = sqlite3.connect(":memory:")
viviendas.to_sql("viviendas", con, index=False, if_exists="replace")

print(f"Entorno preparado: {len(viviendas)} viviendas.")

## 2. Seleccionar con SQL las variables del problema

Queremos estimar `precio_miles_usd` a partir de `superficie_m2`.

- **Variable predictora \(X\):** superficie de la vivienda.
- **Variable objetivo \(Y\):** precio de la vivienda.

Usá SQL para recuperar únicamente esas dos variables.

In [ ]:
consulta = """
SELECT
    superficie_m2,
    precio_miles_usd
FROM viviendas
ORDER BY superficie_m2
"""
df = pd.read_sql_query(consulta, con)
df.head(10)

## 3. Observar la relación antes de modelar

Antes de calcular una correlación o ajustar una recta, mirá el diagrama de dispersión.

Preguntate:

- ¿la nube de puntos muestra una tendencia?
- ¿la relación parece positiva o negativa?
- ¿los puntos siguen exactamente una recta?

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(df["superficie_m2"], df["precio_miles_usd"], alpha=0.75)
plt.xlabel("Superficie (m²)")
plt.ylabel("Precio (miles de USD)")
plt.title("Superficie y precio de las viviendas")
plt.grid(alpha=0.2)
plt.show()

## 4. Medir la asociación lineal: correlación de Pearson

El coeficiente de correlación de Pearson \(r\) resume la **dirección** y la **intensidad de una asociación lineal** entre dos variables cuantitativas.

- \(r\) cercano a **+1**: asociación lineal positiva fuerte.
- \(r\) cercano a **−1**: asociación lineal negativa fuerte.
- \(r\) cercano a **0**: poca asociación lineal.

> **Importante:** una correlación alta no demuestra que una variable cause a la otra.

In [ ]:
r = df["superficie_m2"].corr(df["precio_miles_usd"])
print(f"Correlación de Pearson: r = {r:.3f}")

**Interpretación:** si el valor obtenido es positivo y alto, significa que en estos datos las viviendas de mayor superficie **tienden** a presentar precios mayores. Todavía no estamos afirmando causalidad ni diciendo que la relación sea perfecta.

## 5. Ajustar una regresión lineal simple

La regresión lineal simple representa la relación mediante una recta:

\[
\hat{y} = b_0 + b_1x
\]

donde:

- \(b_0\) es el **intercepto**;
- \(b_1\) es la **pendiente**;
- \(\hat{y}\) es el valor que el modelo **estima** para \(Y\).

In [ ]:
X = df[["superficie_m2"]]
y = df["precio_miles_usd"]

modelo = LinearRegression()
modelo.fit(X, y)

intercepto = modelo.intercept_
pendiente = modelo.coef_[0]

print(f"Intercepto: {intercepto:.2f}")
print(f"Pendiente:  {pendiente:.3f}")
print()
print(
    f"Interpretación de la pendiente: por cada 1 m² adicional, "
    f"el modelo estima un aumento promedio de {pendiente:.3f} miles de USD."
)

La pendiente indica cuánto cambia **el valor estimado por el modelo** cuando la superficie aumenta una unidad. No debe interpretarse automáticamente como un efecto causal.

## 6. Ver la recta de regresión y generar una estimación

In [ ]:
df["predicho"] = modelo.predict(X)

plt.figure(figsize=(7, 4.5))
plt.scatter(df["superficie_m2"], df["precio_miles_usd"], alpha=0.7, label="Observado")
plt.plot(df["superficie_m2"], df["predicho"], linewidth=2, label="Recta de regresión")
plt.xlabel("Superficie (m²)")
plt.ylabel("Precio (miles de USD)")
plt.title("Regresión lineal simple")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

superficie_nueva = pd.DataFrame({"superficie_m2": [100]})
pred_100 = modelo.predict(superficie_nueva)[0]
print(f"Para una vivienda de 100 m², el modelo estima ≈ {pred_100:.1f} miles de USD.")

## 7. Valores observados, predichos y residuos

Un **residuo** es la diferencia entre el valor observado y el valor estimado:

\[
e_i = y_i - \hat{y}_i
\]

- residuo **positivo**: el modelo estimó por debajo del valor observado;
- residuo **negativo**: el modelo estimó por encima del valor observado.

In [ ]:
df["residuo"] = df["precio_miles_usd"] - df["predicho"]

df[["superficie_m2", "precio_miles_usd", "predicho", "residuo"]].head(10).round(2)

## 8. Una primera medida del ajuste: \(R^2\)

El coeficiente de determinación \(R^2\) indica qué proporción de la variabilidad observada en \(Y\) queda representada por el modelo lineal, dentro del conjunto de datos analizado.

En esta regresión lineal simple con intercepto, existe además una conexión útil:

\[
R^2 = r^2
\]

Esto vincula directamente la correlación de Pearson con el ajuste de la recta.

In [ ]:
r2 = r2_score(y, df["predicho"])

print(f"R² del modelo: {r2:.3f}")
print(f"r² de Pearson: {r**2:.3f}")

**Cómo leerlo:** un \(R^2\) mayor indica que la recta representa mejor la variabilidad de los valores observados, pero **no demuestra causalidad ni garantiza que el modelo sea adecuado para cualquier uso**.

También conviene mirar el gráfico y los residuos: una única métrica nunca reemplaza la interpretación del problema.

## 9. Cierre de la actividad

Al finalizar, deberías poder explicar:

- qué diferencia un problema de **regresión** de uno de clasificación;
- qué informa la **correlación de Pearson**;
- qué representan la **pendiente** y el **intercepto**;
- qué diferencia existe entre un valor **observado** y uno **predicho**;
- qué es un **residuo**;
- qué información introductoria aporta **\(R^2\)**.

### Dos límites que no debemos olvidar

1. **Correlación no implica causalidad.**
2. Una recta ajustada al rango observado no debería utilizarse para **extrapolar sin criterio** muy por fuera de esos datos.

En la **Actividad 2** vas a retomar el modelo de clasificación trabajado en la Unidad N° 5 para aprender a evaluar sus aciertos y errores mediante una matriz de confusión y métricas de clasificación.

In [ ]:
con.close()